# Clase `Matriz`: operaciones y validaciones

Esta solución implementa una clase para construir, visualizar y operar matrices. Incluye:

- Entrada mediante listas, texto o `input()`.
- Las cuatro operaciones aritméticas entre matrices: suma, resta, producto y división elemento a elemento.
- Producto matricial, transposición y multiplicación por un escalar.
- Las tres operaciones elementales por filas.
- Mensajes claros cuando una operación no es posible.
- Matrices no numéricas para visualización y transposición.

In [1]:
from numbers import Number


class Matriz:
    """Representa una matriz rectangular y sus operaciones principales."""

    def __init__(self, datos):
        if not isinstance(datos, (list, tuple)) or not datos:
            raise ValueError("La matriz debe contener al menos una fila.")

        filas = []
        for fila in datos:
            if not isinstance(fila, (list, tuple)) or not fila:
                raise ValueError("Cada fila debe contener al menos un elemento.")
            filas.append(list(fila))

        columnas = len(filas[0])
        if any(len(fila) != columnas for fila in filas):
            raise ValueError("Todas las filas deben tener la misma longitud.")

        self.datos = filas
        self.filas = len(filas)
        self.columnas = columnas

    @staticmethod
    def _convertir_valor(valor):
        valor = valor.strip()
        try:
            return int(valor)
        except ValueError:
            try:
                return float(valor)
            except ValueError:
                return valor

    @classmethod
    def desde_texto(cls, texto):
        """Crea una matriz: separa columnas con comas y filas con punto y coma."""
        if not texto or not texto.strip():
            raise ValueError("La entrada de la matriz no puede estar vacía.")
        filas = [
            [cls._convertir_valor(valor) for valor in fila.split(",")]
            for fila in texto.split(";")
        ]
        return cls(filas)

    @classmethod
    def desde_usuario(cls):
        print("Separa columnas con comas y filas con punto y coma.")
        print("Ejemplo: 1,2,3;4,5,6")
        return cls.desde_texto(input("Ingresa la matriz: "))

    @property
    def forma(self):
        return self.filas, self.columnas

    def copia(self):
        return Matriz([fila[:] for fila in self.datos])

    def mostrar(self, titulo="Matriz"):
        anchos = [
            max(len(str(self.datos[fila][columna])) for fila in range(self.filas))
            for columna in range(self.columnas)
        ]
        print(f"{titulo} ({self.filas}x{self.columnas})")
        for fila in self.datos:
            contenido = "  ".join(
                str(valor).rjust(anchos[indice]) for indice, valor in enumerate(fila)
            )
            print(f"│ {contenido} │")

    def _validar_matriz(self, otra):
        if not isinstance(otra, Matriz):
            raise TypeError("La operación requiere otra instancia de Matriz.")

    def _validar_misma_forma(self, otra):
        self._validar_matriz(otra)
        if self.forma != otra.forma:
            raise ValueError(
                f"Las matrices deben tener la misma forma: {self.forma} != {otra.forma}."
            )

    def _validar_numerica(self, nombre="La matriz"):
        if any(not isinstance(valor, Number) for fila in self.datos for valor in fila):
            raise TypeError(f"{nombre} debe contener únicamente valores numéricos.")

    def _validar_binaria_numerica(self, otra, misma_forma=True):
        if misma_forma:
            self._validar_misma_forma(otra)
        else:
            self._validar_matriz(otra)
        self._validar_numerica("La primera matriz")
        otra._validar_numerica("La segunda matriz")

    def _mostrar_operacion(self, simbolo, otra, resultado):
        self.mostrar("A")
        print(simbolo)
        otra.mostrar("B")
        print("=")
        resultado.mostrar("Resultado")

    def _operacion_elemento(self, otra, funcion, simbolo, mostrar=True):
        self._validar_binaria_numerica(otra)
        resultado = Matriz([
            [funcion(a, b) for a, b in zip(fila_a, fila_b)]
            for fila_a, fila_b in zip(self.datos, otra.datos)
        ])
        if mostrar:
            self._mostrar_operacion(simbolo, otra, resultado)
        return resultado

    def sumar(self, otra, mostrar=True):
        return self._operacion_elemento(otra, lambda a, b: a + b, "+", mostrar)

    def restar(self, otra, mostrar=True):
        return self._operacion_elemento(otra, lambda a, b: a - b, "−", mostrar)

    def multiplicar_elemento(self, otra, mostrar=True):
        return self._operacion_elemento(otra, lambda a, b: a * b, "× elemento", mostrar)

    def dividir_elemento(self, otra, mostrar=True):
        self._validar_binaria_numerica(otra)
        if any(valor == 0 for fila in otra.datos for valor in fila):
            raise ZeroDivisionError("No se puede dividir entre una matriz que contenga ceros.")
        return self._operacion_elemento(otra, lambda a, b: a / b, "÷ elemento", mostrar)

    def producto_matricial(self, otra, mostrar=True):
        self._validar_binaria_numerica(otra, misma_forma=False)
        if self.columnas != otra.filas:
            raise ValueError(
                "Para multiplicar matrices, las columnas de la primera deben coincidir "
                "con las filas de la segunda."
            )
        resultado = Matriz([
            [
                sum(self.datos[i][k] * otra.datos[k][j] for k in range(self.columnas))
                for j in range(otra.columnas)
            ]
            for i in range(self.filas)
        ])
        if mostrar:
            self._mostrar_operacion("@", otra, resultado)
        return resultado

    def multiplicar_escalar(self, escalar, mostrar=True):
        self._validar_numerica()
        if not isinstance(escalar, Number):
            raise TypeError("El escalar debe ser numérico.")
        resultado = Matriz([[valor * escalar for valor in fila] for fila in self.datos])
        if mostrar:
            self.mostrar("A")
            print(f"× escalar {escalar} =")
            resultado.mostrar("Resultado")
        return resultado

    def transponer(self, mostrar=True):
        resultado = Matriz([
            [self.datos[fila][columna] for fila in range(self.filas)]
            for columna in range(self.columnas)
        ])
        if mostrar:
            self.mostrar("A")
            print("Transpuesta =")
            resultado.mostrar("Aᵀ")
        return resultado

    def intercambiar_filas(self, fila_1, fila_2, mostrar=True):
        resultado = self.copia()
        try:
            resultado.datos[fila_1], resultado.datos[fila_2] = (
                resultado.datos[fila_2], resultado.datos[fila_1]
            )
        except IndexError as error:
            raise IndexError("El índice de una fila está fuera de rango.") from error
        if mostrar:
            print(f"Intercambio F{fila_1 + 1} ↔ F{fila_2 + 1}")
            resultado.mostrar("Resultado")
        return resultado

    def escalar_fila(self, fila, escalar, mostrar=True):
        self._validar_numerica()
        if not isinstance(escalar, Number) or escalar == 0:
            raise ValueError("El escalar debe ser un número distinto de cero.")
        if not 0 <= fila < self.filas:
            raise IndexError("El índice de la fila está fuera de rango.")
        resultado = self.copia()
        resultado.datos[fila] = [valor * escalar for valor in resultado.datos[fila]]
        if mostrar:
            print(f"F{fila + 1} ← {escalar}·F{fila + 1}")
            resultado.mostrar("Resultado")
        return resultado

    def sumar_multiplo_fila(self, origen, destino, escalar, mostrar=True):
        self._validar_numerica()
        if not isinstance(escalar, Number):
            raise TypeError("El escalar debe ser numérico.")
        if not (0 <= origen < self.filas and 0 <= destino < self.filas):
            raise IndexError("El índice de una fila está fuera de rango.")
        if origen == destino:
            raise ValueError("La fila de origen y la de destino deben ser diferentes.")
        resultado = self.copia()
        resultado.datos[destino] = [
            destino_valor + escalar * origen_valor
            for origen_valor, destino_valor in zip(
                self.datos[origen], self.datos[destino]
            )
        ]
        if mostrar:
            print(f"F{destino + 1} ← F{destino + 1} + {escalar}·F{origen + 1}")
            resultado.mostrar("Resultado")
        return resultado

    def __add__(self, otra):
        return self.sumar(otra)

    def __sub__(self, otra):
        return self.restar(otra)

    def __mul__(self, otra):
        if isinstance(otra, Number):
            return self.multiplicar_escalar(otra)
        return self.multiplicar_elemento(otra)

    def __truediv__(self, otra):
        return self.dividir_elemento(otra)

    def __matmul__(self, otra):
        return self.producto_matricial(otra)


## Creación y visualización

El usuario puede escribir `1,2;3,4`, donde la coma separa columnas y el punto y coma separa filas.

In [2]:
A = Matriz.desde_texto("2,4;6,8")
B = Matriz([[1, 2], [3, 4]])

A.mostrar("Matriz A")
B.mostrar("Matriz B")

Matriz A (2x2)
│ 2  4 │
│ 6  8 │
Matriz B (2x2)
│ 1  2 │
│ 3  4 │


## Cuatro operaciones aritméticas entre matrices

In [3]:
suma = A.sumar(B)
print("\n" + "─" * 24 + "\n")
resta = A.restar(B)
print("\n" + "─" * 24 + "\n")
producto_elemento = A.multiplicar_elemento(B)
print("\n" + "─" * 24 + "\n")
division_elemento = A.dividir_elemento(B)

A (2x2)
│ 2  4 │
│ 6  8 │
+
B (2x2)
│ 1  2 │
│ 3  4 │
=
Resultado (2x2)
│ 3   6 │
│ 9  12 │

────────────────────────

A (2x2)
│ 2  4 │
│ 6  8 │
−
B (2x2)
│ 1  2 │
│ 3  4 │
=
Resultado (2x2)
│ 1  2 │
│ 3  4 │

────────────────────────

A (2x2)
│ 2  4 │
│ 6  8 │
× elemento
B (2x2)
│ 1  2 │
│ 3  4 │
=
Resultado (2x2)
│  2   8 │
│ 18  32 │

────────────────────────

A (2x2)
│ 2  4 │
│ 6  8 │
÷ elemento
B (2x2)
│ 1  2 │
│ 3  4 │
=
Resultado (2x2)
│ 2.0  2.0 │
│ 2.0  2.0 │


## Producto matricial, transposición y escalar

In [4]:
producto_matricial = A.producto_matricial(B)
print("\n" + "─" * 24 + "\n")
transpuesta = A.transponer()
print("\n" + "─" * 24 + "\n")
por_escalar = A.multiplicar_escalar(3)

A (2x2)
│ 2  4 │
│ 6  8 │
@
B (2x2)
│ 1  2 │
│ 3  4 │
=
Resultado (2x2)
│ 14  20 │
│ 30  44 │

────────────────────────

A (2x2)
│ 2  4 │
│ 6  8 │
Transpuesta =
Aᵀ (2x2)
│ 2  6 │
│ 4  8 │

────────────────────────

A (2x2)
│ 2  4 │
│ 6  8 │
× escalar 3 =
Resultado (2x2)
│  6  12 │
│ 18  24 │


## Operaciones elementales por filas

Se muestran las tres operaciones estándar: intercambio, escalamiento y suma de un múltiplo de otra fila.

In [5]:
intercambio = A.intercambiar_filas(0, 1)
print()
fila_escalada = A.escalar_fila(0, 2)
print()
suma_de_filas = A.sumar_multiplo_fila(origen=0, destino=1, escalar=-3)

Intercambio F1 ↔ F2
Resultado (2x2)
│ 6  8 │
│ 2  4 │

F1 ← 2·F1
Resultado (2x2)
│ 4  8 │
│ 6  8 │

F2 ← F2 + -3·F1
Resultado (2x2)
│ 2   4 │
│ 0  -4 │


## Validación y mensajes de error

Los errores se capturan en el ejemplo para que el notebook continúe ejecutándose.

In [6]:
pruebas_invalidas = [
    ("Suma con dimensiones diferentes", lambda: A.sumar(Matriz([[1, 2, 3]]))),
    ("División entre cero", lambda: A.dividir_elemento(Matriz([[1, 0], [1, 1]]))),
    ("Producto matricial incompatible", lambda: A.producto_matricial(Matriz([[1, 2, 3]]))),
]

for descripcion, operacion in pruebas_invalidas:
    try:
        operacion()
    except (TypeError, ValueError, ZeroDivisionError) as error:
        print(f"{descripcion}: {error}")

Suma con dimensiones diferentes: Las matrices deben tener la misma forma: (2, 2) != (1, 3).
División entre cero: No se puede dividir entre una matriz que contenga ceros.
Producto matricial incompatible: Para multiplicar matrices, las columnas de la primera deben coincidir con las filas de la segunda.


## Matriz no numérica

Las matrices de texto pueden visualizarse y transponerse. Las operaciones aritméticas muestran un error descriptivo.

In [7]:
personas = Matriz([
    ["Nombre", "Ciudad"],
    ["Ana", "Tijuana"],
    ["Luis", "Mexicali"],
])

personas.mostrar("Matriz de texto")
print()
personas.transponer()

try:
    personas.sumar(personas)
except TypeError as error:
    print(f"\nOperación numérica rechazada correctamente: {error}")

Matriz de texto (3x2)
│ Nombre    Ciudad │
│    Ana   Tijuana │
│   Luis  Mexicali │

A (3x2)
│ Nombre    Ciudad │
│    Ana   Tijuana │
│   Luis  Mexicali │
Transpuesta =
Aᵀ (2x3)
│ Nombre      Ana      Luis │
│ Ciudad  Tijuana  Mexicali │

Operación numérica rechazada correctamente: La primera matriz debe contener únicamente valores numéricos.
